In [ ]:
import os
from configs import openai_token, together_ai_token

os.environ["OPENAI_API_KEY"] = openai_token
# Replace with your model provider, we use Anyscale's Mixtral here.
os.environ["TOGETHERAI_API_KEY"] = together_ai_token

from routellm.controller import Controller


client = Controller(
  routers=["mf"],
  strong_model="together_ai/meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
  weak_model="together_ai/mistralai/Mixtral-8x7B-Instruct-v0.1",
  # The config for the router (best-performing config by default)
  config = {
    "mf": {
      "checkpoint_path": "routellm/mf_gpt4_augmented"
    }
  },
  # Override API base and key for LLM calls
  api_base=None,
  api_key=None,
  # Display a progress bar for operations
  progress_bar=True,
)

Loading mf:   0%|          | 0/1 [00:00<?, ?it/s]


In [15]:
response = client.chat.completions.create(
  # This tells RouteLLM to use the MF router with a cost threshold of 0.11593
  model="router-mf-0.11593",
  messages=[
    {"role": "user", "content": "Hello!"}
  ]
)
print(response.choices[0]["message"]["content"])

 Hello! How can I help you today? Is there a particular topic you would like to discuss or learn about related to data science? I'm here to answer any questions you might have.


In [23]:
from datasets import load_dataset

ds = load_dataset("cais/mmlu", "all")

Generating auxiliary_train split: 100%|██████████| 99842/99842 [00:01<00:00, 93325.42 examples/s] 


In [38]:
from config import hf_token
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Idavidrein/gpqa", "gpqa_main", token=hf_token)

Generating train split: 100%|██████████| 448/448 [00:00<00:00, 6041.06 examples/s]


In [46]:
ds["train"][0]["Question"]

"A large gene has dozens of exons, of which the central ones code for folded triple helical repeats that connect the cytoskeleton with sarcolemma and extracellular space. Each exon usually codes for one folded triple alpha helix. The most common mutations of the gene are central exon deletions that create out-of-frame peptides and progressive degenerative organ waste. A solution is to deliver a Morpholino that recognizes the 5' end of the out-of-frame exon in pre-mRNA. The molecule prevents binding of the spliceosome and creates exon skipping and in-frame joining. Several missing exons are well tolerated by an organism. Which structure below is not involved in the proposed therapy?"

In [ ]:
ds["train"][0]["Question"]



dict_keys(['Pre-Revision Question', 'Pre-Revision Correct Answer', 'Pre-Revision Incorrect Answer 1', 'Pre-Revision Incorrect Answer 2', 'Pre-Revision Incorrect Answer 3', 'Pre-Revision Explanation', 'Self-reported question-writing time (minutes)', 'Question', 'Correct Answer', 'Incorrect Answer 1', 'Incorrect Answer 2', 'Incorrect Answer 3', 'Explanation', 'Revision Comments (from Question Writer)', 'Subdomain', "Writer's Difficulty Estimate", 'Extra Revised Question', 'Extra Revised Explanation', 'Extra Revised Correct Answer', 'Extra Revised Incorrect Answer 1', 'Extra Revised Incorrect Answer 2', 'Extra Revised Incorrect Answer 3', 'Non-Expert Validator Accuracy', 'Majority Non-Expert Vals Incorrect', 'Expert Validator Accuracy', 'Record ID', 'High-level domain', 'Question Writer', 'Feedback_EV_1', 'Validator Revision Suggestion_EV_1', 'Is First Validation_EV_1', 'Post hoc agreement_EV_1', 'Sufficient Expertise?_EV_1', 'Understand the question?_EV_1', 'Question Difficulty_EV_1', 'V

In [50]:
questions = []
for data_row in ds["train"]:
    choices = []
    for i in range(1, 4):
        choices.append(data_row[f"Incorrect Answer {i}"])
    choices.append(f"{data_row['Correct Answer']}")
    q_str = f"{data_row['Question']}\n"
    for i, choice in enumerate(choices):
        q_str += f"{chr(ord('a') + i)}){choice}\n"
    questions.append(q_str)
    

In [57]:
response = client.chat.completions.create(
  # This tells RouteLLM to use the MF router with a cost threshold of 0.11593
  model="router-mf-0.11593",
  messages=[
      {
          "role": "system",
          "content": "You are a multiple-choice answering assistant. Given a question with multiple-choice options, your task is to only reply with the single correct option's letter (e.g., A, B, C, or D). Do not explain your answer or provide any additional text."
          },
    {"role": "user", "content": f"{questions[0]}"}
  ]
)
print(response.choices[0]["message"]["content"])

D


In [58]:
train_size = int(len(questions) * 0.8)
test_data = questions[train_size:]

In [62]:
import tqdm
answers = []
for question in tqdm.tqdm(test_data):
    response = client.chat.completions.create(
    # This tells RouteLLM to use the MF router with a cost threshold of 0.11593
    model="router-mf-0.11593",
    messages=[
        {
            "role": "system",
            "content": "You are a multiple-choice answering assistant. Given a question with multiple-choice options, your task is to only reply with the single correct option's letter (e.g., A, B, C, or D). Do not explain your answer or provide any additional text."
            },
        {"role": "user", "content": f"{question}"}
    ]
    )
    answers.append(response.choices[0]["message"]["content"])

 54%|█████▍    | 49/90 [07:08<05:58,  8.75s/it]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



RateLimitError: litellm.RateLimitError: RateLimitError: Together_aiException - You have reached the rate limit specific to this model meta-llama/Llama-3.3-70B-Instruct-Turbo-Free. The maximum rate limit for this model is 10.0 queries and 180000 tokens per minute. This limit differs from the general rate limits published at Together AI rate limits documentation (https://docs.together.ai/docs/rate-limits). For inquiries about increasing your model-specific rate limit, please contact our sales team (https://www.together.ai/forms/contact-sales)

In [ ]:
trimmed_ans = [ans.strip().lower() for ans in answers]

In [67]:
import numpy as np
np.array(trimmed_ans) == "d"

array([False, False,  True,  True, False, False,  True, False,  True,
       False, False,  True, False,  True, False, False,  True,  True,
       False,  True, False,  True, False,  True, False, False,  True,
        True,  True,  True,  True, False, False, False,  True, False,
        True,  True,  True, False,  True,  True, False, False,  True,
       False,  True,  True, False])

In [ ]:
client = Controller(
  routers=["mf"],
  strong_model="together_ai/meta-llama/Llama-3.3-70B-Instruct-Turbo",
  weak_model="together_ai/mistralai/Mixtral-8x7B-Instruct-v0.1",
  # The config for the router (best-performing config by default)
  config = {
    "mf": {
      "checkpoint_path": "routellm/mf_gpt4_augmented"
    }
  },
  # Override API base and key for LLM calls
  api_base=None,
  api_key=None,
  # Display a progress bar for operations
  progress_bar=True,
)

Loading mf:   0%|          | 0/1 [00:00<?, ?it/s]


In [69]:
for question in tqdm.tqdm(test_data[len(trimmed_ans):]):
    response = client.chat.completions.create(
    # This tells RouteLLM to use the MF router with a cost threshold of 0.11593
    model="router-mf-0.11593",
    messages=[
        {
            "role": "system",
            "content": "You are a multiple-choice answering assistant. Given a question with multiple-choice options, your task is to only reply with the single correct option's letter (e.g., A, B, C, or D). Do not explain your answer or provide any additional text."
            },
        {"role": "user", "content": f"{question}"}
    ]
    )
    answers.append(response.choices[0]["message"]["content"])

100%|██████████| 41/41 [00:36<00:00,  1.12it/s]


In [75]:
trimmed_ans = [ans.strip().lower() for ans in answers]

In [76]:
print((np.array(trimmed_ans) == "d").mean())

0.4


In [78]:
client = Controller(
  routers=["mf"],
  strong_model="together_ai/meta-llama/Llama-3.3-70B-Instruct-Turbo",
  weak_model="together_ai/google/gemma-2-27b-it",
  # The config for the router (best-performing config by default)
  config = {
    "mf": {
      "checkpoint_path": "routellm/mf_gpt4_augmented"
    }
  },
  # Override API base and key for LLM calls
  api_base=None,
  api_key=None,
  # Display a progress bar for operations
  progress_bar=True,
)

answers2 = []
for question in tqdm.tqdm(test_data):
    response = client.chat.completions.create(
    # This tells RouteLLM to use the MF router with a cost threshold of 0.11593
    model="router-mf-0.11593",
    messages=[
        {
            "role": "system",
            "content": "You are a multiple-choice answering assistant. Given a question with multiple-choice options, your task is to only reply with the single correct option's letter (e.g., A, B, C, or D). Do not explain your answer or provide any additional text."
            },
        {"role": "user", "content": f"{question}"}
    ]
    )
    answers2.append(response.choices[0]["message"]["content"])

100%|██████████| 90/90 [01:02<00:00,  1.45it/s]


In [80]:
trimmed_ans = np.array([ans.strip().lower() for ans in answers2])
print((trimmed_ans == "d").mean())


0.3888888888888889


In [83]:
client = Controller(
  routers=["mf"],
  strong_model="together_ai/meta-llama/Llama-3.3-70B-Instruct-Turbo",
  weak_model="together_ai/Qwen/Qwen2.5-72B-Instruct-Turbo",
  # The config for the router (best-performing config by default)
  config = {
    "mf": {
      "checkpoint_path": "routellm/mf_gpt4_augmented"
    }
  },
  # Override API base and key for LLM calls
  api_base=None,
  api_key=None,
  # Display a progress bar for operations
  progress_bar=True,
)

answers3 = []
for question in tqdm.tqdm(test_data):
    response = client.chat.completions.create(
    # This tells RouteLLM to use the MF router with a cost threshold of 0.11593
    model="router-mf-0.11593",
    messages=[
        {
            "role": "system",
            "content": "You are a multiple-choice answering assistant. Given a question with multiple-choice options, your task is to only reply with the single correct option's letter (e.g., A, B, C, or D). Do not explain your answer or provide any additional text."
            },
        {"role": "user", "content": f"{question}"}
    ]
    )
    answers3.append(response.choices[0]["message"]["content"])

trimmed_ans = np.array([ans.strip().lower() for ans in answers3])
print((trimmed_ans == "d").mean())


100%|██████████| 90/90 [01:06<00:00,  1.36it/s]

0.36666666666666664


In [ ]:
answers3

['d',
 'b',
 'd',
 'a',
 'c',
 'c',
 'a',
 'b',
 'D',
 'a',
 'a',
 'a',
 'a',
 'b',
 'a',
 'd',
 'd',
 'a',
 'b',
 'b',
 'a',
 'd',
 'c',
 'd',
 'A',
 'a',
 'c',
 'D',
 'd',
 'b',
 'd',
 'b',
 'c',
 'a',
 'd',
 'a',
 'b',
 'b',
 'd',
 'b',
 'd',
 'b',
 'c',
 'b',
 'b',
 'c',
 'b',
 'd',
 'd',
 'a',
 'b',
 'd',
 'a',
 'b',
 'a',
 'b',
 'b',
 'b',
 'D',
 'b',
 'd',
 'c',
 'c',
 'a',
 'a',
 'a',
 'c',
 'd',
 'a',
 'c',
 'C',
 'A',
 'c',
 'c',
 'a',
 'd',
 'a',
 'd',
 'c',
 'b',
 'a',
 'a',
 'B',
 'a',
 'a',
 'a',
 'b',
 'a',
 'b',
 'c']